# Tranche A on a free Colab T4**Turkish -> Azerbaijani transfer mechanisms** - DLE-AI-202 Track 1.Separates syntactic transfer (M1), lexical warm-up (M2) and tokenizerquality (M3) behind the Kardes-NLU result.## What this notebook is forA free T4 is **not a grid machine**. The full 114-run grid projects to**81-108 h** on a T4 (`docs/COMPUTE_ESTIMATES.md`), far beyond any freesession, which also disconnects on idle. So this notebook does the threethings a T4 *can* do, in order:1. **Verify** - pinned environment, frozen-config hashes, test suite, queue shape.2. **Build** - splits, scrambled Turkish, the OMP transplant and its placebos,   and the C1/C1c/top-1 controls that are the only checks able to catch a   broken transplant.3. **Tranche A** - `xlm15`, n=2000, `baza` vs `tokenizator`, 5 seeds. Ten runs,   **no Turkish stage**, so no Phase-1 cache is needed. This is the cheapest   tranche that answers the question everything else depends on.Tranche A is **6.3-8.4 h on a T4** - two to three free sessions. Step 4 puts`results/` and the checkpoint cache on Drive, so `run.skip_existing` resumesexactly where a dropped session stopped.## What Tranche A decides| Outcome | Reading | Action ||---|---|---|| Both conditions escape in most seeds (>= 3/5) | Pipeline works, task is learnable | Proceed to B, then C, then D || `tokenizator` escapes reliably, `baza` does not | **Strong M3 evidence** | Proceed - this is a headline result || Neither escapes in any seed | n=2000 is below the detectability threshold | **Stop and re-plan** at n=10000. Do *not* tune the optimizer || Escape is erratic, uncorrelated with condition | Seed variance dominates | Report escape rate as the primary outcome; raise seed count |These rules are fixed **before** the data (run plan section 8) so the outcomecannot be reinterpreted afterwards. A negative result is a valid deliverable.> **Do not change `training.batch_size` or `training.grad_accum` here.** They> are hash-locked, and a run under different settings is not comparable to the> rest of the grid - it quarantines itself. See `configs/FROZEN.md`.

## 1-5 - Setup and verification

In [ ]:
# ---------------------------------------------------------------- 1. the repo
# Three ways to get the code in. Set ONE of these and run.
REPO_URL   = ""          # e.g. "https://github.com/<user>/az-tokenizer-transfer.git"
DRIVE_ZIP  = ""          # e.g. "/content/drive/MyDrive/az-tokenizer-transfer.zip"
UPLOAD_ZIP = True        # otherwise: prompt for a local .zip upload

import os, shutil, subprocess, sys, zipfile
from pathlib import Path

WORK = Path("/content/project")

if REPO_URL:
    if WORK.exists(): shutil.rmtree(WORK)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)], check=True)
elif DRIVE_ZIP:
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP) as z: z.extractall(WORK)
elif UPLOAD_ZIP:
    from google.colab import files
    up = files.upload()
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(next(iter(up))) as z: z.extractall(WORK)

# The zip may contain a single top-level folder; descend into it.
if not (WORK / "configs" / "experiment.yaml").exists():
    inner = [p for p in WORK.iterdir() if p.is_dir() and (p / "configs").exists()]
    if len(inner) == 1: WORK = inner[0]

assert (WORK / "configs" / "experiment.yaml").exists(), f"repo not found under {WORK}"
os.chdir(WORK); sys.path.insert(0, str(WORK))
print("project root:", WORK)
print(sorted(p.name for p in WORK.iterdir())[:14])

In [ ]:
# ---------------------------------------------------------------- 2. pinned environment
# requirements.lock is the exact stack the results were produced on. Version
# drift between machines is precisely what invalidates a cross-machine
# comparison, so we install the lock rather than letting pip resolve fresh.
#
# This REPLACES Colab's preinstalled torch/numpy and therefore REQUIRES a
# runtime restart afterwards. Expect 5-9 minutes.
!pip install -q -r requirements.lock --extra-index-url https://download.pytorch.org/whl/cu121

print("\n" + "=" * 66)
print("RESTART THE RUNTIME NOW (Runtime -> Restart session), then run the")
print("next cell. Do NOT re-run the cells above after restarting.")
print("=" * 66)

In [ ]:
# ---------------------------------------------------------------- 3. after the restart
import os, sys
from pathlib import Path
WORK = Path("/content/project")
if not (WORK / "configs").exists():
    WORK = next(p for p in WORK.iterdir() if (p / "configs").exists())
os.chdir(WORK); sys.path.insert(0, str(WORK))

import torch, transformers, numpy
print("cwd        ", Path.cwd())
print("torch      ", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("numpy      ", numpy.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu        ", p.name, f"{p.total_memory/1e9:.1f} GB")
else:
    print("*** NO GPU. Runtime -> Change runtime type -> GPU.")

In [ ]:
# ---------------------------------------------------------------- 4. persistence
# Colab sessions end without warning. `run.skip_existing` resumes from durable
# result files, so pointing results/ and the Turkish cache at Drive turns a
# dropped session from lost work into a pause.
USE_DRIVE = True

import os, shutil
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = Path("/content/drive/MyDrive/az-tokenizer-transfer")
    for sub in ("results", "artifacts"):
        (STORE / sub).mkdir(parents=True, exist_ok=True)
        local = Path.cwd() / sub
        if local.is_symlink(): local.unlink()
        elif local.exists():
            # Preserve anything already committed in the repo (results/*.json).
            for item in local.iterdir():
                target = STORE / sub / item.name
                if not target.exists():
                    (shutil.copytree if item.is_dir() else shutil.copy2)(item, target)
            shutil.rmtree(local)
        local.symlink_to(STORE / sub, target_is_directory=True)
    print("results/ and artifacts/ -> ", STORE)
else:
    print("Ephemeral storage: everything is lost when the session ends.")

import shutil as _sh
free = _sh.disk_usage(Path.cwd()).free / 1e9
print(f"free disk: {free:.1f} GB")

In [ ]:
# ---------------------------------------------------------------- 5. verification gate
# Nothing is launched until the frozen config matches its hash lock, the test
# suite is green, and the queue has the expected shape. Each of these has
# caught a real defect in this project.
!python -c "from src.utils import verify_config_hashes; verify_config_hashes(); print('frozen config hashes OK')"
!python -m pytest tests/ -q -m "not slow" --no-header
!python -m src.training.run_grid --config configs/experiment.yaml --dry-run | head -3

## 6 - DataDeduplication happens **before** the split. Skipping that put identical textsin both train and test in an earlier revision - hidden test-set leakage, whichthe brief lists as an automatic deduction. `splits.py` asserts zero overlapacross the three splits and exits if it finds any.

In [ ]:
!python -m src.data.splits     --config configs/experiment.yaml
!python -m src.data.scramble   --config configs/experiment.yaml
!python -m src.data.truncation --config configs/experiment.yaml

import json
s = json.load(open("results/splits.json", encoding="utf-8"))
print("AZ  train/val/test:", s["az"]["train"], s["az"]["val"], s["az"]["test"])
print("AZ  labels        :", s["az"]["label_map"])
print("dedup             :", s["az"]["dedup"]["duplicate_pct"], "% dup,",
      s["az"]["dedup"]["conflicting_texts"], "conflicting")
print("leakage           :", s["leakage_check"])
assert not any(s["leakage_check"].values()), "TEST-SET LEAKAGE"

## 7 - Transplant and its controlsOne transplant **per base model** - the embedding matrix lives in that base'sown space, so a shared directory would load weights of the wrong shape.The canonical artifact is the **rescaled** variant. The controls below measurethat artifact, not the no-rescale ablation baseline; conflating the two is adefect this pass fixed.**Read `top1_*` before believing any BPC number.** A lower BPC with zero top-1accuracy is a calibration change, not restored masked-LM ability. Bothdiagnostics now draw the same corpus from `az_train.jsonl` (~300 sentences,~1,300 masked positions) instead of six hard-coded sentences, which gave 12masked positions and could not distinguish a destroyed transplant from amerely damaged one.

In [ ]:
!python -m src.transplant.build                  --config configs/experiment.yaml
!python -m src.transplant.build_rescale_variant  --config configs/experiment.yaml
!python -m src.transplant.build --config configs/experiment.yaml --method mean        --tag mean_k64
!python -m src.transplant.build_rescale_variant --config configs/experiment.yaml --method mean
!python -m src.transplant.build --config configs/experiment.yaml --method random_coef --tag random_coef_k64
!python -m src.transplant.build_rescale_variant --config configs/experiment.yaml --method random_coef

In [ ]:
import subprocess, sys
from src.utils import load_config, resolve_bases, transplant_dir, canonical_transplant_tag
cfg = load_config("configs/experiment.yaml", require_complete=False)
tag = canonical_transplant_tag(cfg)
for b in resolve_bases(cfg):
    d = transplant_dir(cfg, b.short, tag)
    print("=" * 60, "\n", b.role, d)
    subprocess.run([sys.executable, "-m", "src.transplant.controls",
                    "--config", "configs/experiment.yaml", "--base", b.role,
                    "--transplanted", str(d), "--device", "cuda"], check=True)

!python -m src.transplant.top1_accuracy       --config configs/experiment.yaml --device cuda
!python -m src.transplant.check_embedding_norms --config configs/experiment.yaml
!python -m src.transplant.cross_base_quality  --config configs/experiment.yaml

In [ ]:
import json
q = json.load(open("results/cross_base_transplant_quality.json", encoding="utf-8"))
for base, r in q["per_base"].items():
    t1b, t1t = r.get("top1_base", {}), r.get("top1_transplanted_canonical", {})
    print(f"{base:6s} dBPC={r.get('C1c_delta_bpc'):>9} norm_ratio={r.get('norm_ratio')} "
          f"healthy={r.get('norm_ratio_healthy')}")
    print(f"       top-1 base {t1b.get('correct')}/{t1b.get('masked')} "
          f"-> transplanted {t1t.get('correct')}/{t1t.get('masked')}")
print("\nverdict:", q["comparison"].get("worse_transplant"))
print("\nIf BOTH transplanted top-1 counts are ~0 on a corpus of ~1,300 masked")
print("positions, the transplant has destroyed masked-LM ability in both bases.")
print("That is a finding to report, and it weakens the xlmr zero-effect control:")
print("a null result there could mean 'no deficit to fix' OR 'transplant broken'.")

## 8 - Tranche ATen runs, no Turkish stage. `--budget-hours` stops cleanly **between** runsrather than being killed mid-write, so a session that runs out of time leavesonly complete results behind.Re-run this cell in a fresh session to continue: completed runs are skipped.

In [ ]:
# Free Colab sessions are unpredictable; 3.5 h leaves margin to run the
# analysis below before a disconnect. Raise it if your session is stable.
BUDGET_HOURS = 3.5
SEEDS = ""   # e.g. "42,1337" to take Tranche A a few seeds at a time

cmd = ("python -m src.training.run_grid --config configs/experiment.yaml "
       f"--tranche A --phase 2 --streams 1 --budget-hours {BUDGET_HOURS}")
if SEEDS:
    cmd += f" --seeds {SEEDS}"
print(cmd)
!{cmd}

In [ ]:
# Progress against the 10 Tranche-A runs.
import json, glob
done = sorted(glob.glob("results/runs/base=xlm15__cond=*__n=2000__seed=*.json"))
print(f"{len(done)} / 10 Tranche-A runs complete\n")
for p in done:
    r = json.load(open(p, encoding="utf-8"))
    if r.get("condition") not in ("baza", "tokenizator"): continue
    print(f"{r['condition']:12s} seed={r['seed']:<5} escaped={str(r['escaped']):5s} "
          f"step={r['escape_step']} f1={r['selected_validation_macro_f1']:.4f} "
          f"{r['runtime_sec']:.0f}s  ({r['per_run_verdict']})")

## 9 - Analysis

In [ ]:
# ---------------------------------------------------------------- analysis
# Every table and figure is regenerated from results/ alone; nothing is read
# from a notebook variable. Empty cells print NOT MEASURED rather than being
# silently dropped.
!python -m src.analysis.aggregate --config configs/experiment.yaml
!python -m src.analysis.stats     --config configs/experiment.yaml
!python -m src.analysis.decompose --config configs/experiment.yaml
!python -m src.analysis.report    --config configs/experiment.yaml

from IPython.display import Markdown, display
display(Markdown(open("results/paper_tables.md", encoding="utf-8").read()))

In [ ]:
from IPython.display import Image, display
for name in ("escape_rate.png", "conditional_macro_f1.png"):
    p = f"figures/{name}"
    print(p); display(Image(p))

## 10 - Apply the decision ruleEscape rate is the primary outcome. Conditional macro-F1 is reported **only**over seeds that escaped - averaging a collapsed run's flat F1 together with alearned run's F1 would merge two different quantities.

In [ ]:
import json
cells = {(c["base"], c["condition"]): c
         for c in json.load(open("results/decompose.json", encoding="utf-8"))["cells"]}
rows = [cells.get(("xlm15", c)) for c in ("baza", "tokenizator")]
if any(r is None or r["status"] == "NOT MEASURED" for r in rows):
    print("Tranche A incomplete - no decision yet.")
else:
    baza, omp = rows
    print(f"baza        escape {baza['escape_rate_count']}  conditional F1 {baza['conditional_macro_f1']}")
    print(f"tokenizator escape {omp['escape_rate_count']}  conditional F1 {omp['conditional_macro_f1']}")
    b, o = baza["n_escaped"], omp["n_escaped"]
    print()
    if b == 0 and o == 0:
        print("NEITHER ESCAPES -> STOP AND RE-PLAN. Move the reference size to")
        print("n=10000 before spending the window. Do NOT tune the optimizer.")
    elif o >= 3 and b < 3:
        print("tokenizator ESCAPES, baza DOES NOT -> strong M3 evidence.")
        print("This is a headline result; proceed to Tranche B.")
    elif b >= 3 and o >= 3:
        print("BOTH ESCAPE -> pipeline works. Proceed to B, then C, then D.")
    else:
        print("ERRATIC -> seed variance may dominate. Report escape rate as the")
        print("primary outcome and raise the seed count at the reference size only.")

In [ ]:
# Measured throughput -> re-project the grid from a real number.
import json, glob, statistics
rt = [json.load(open(p, encoding="utf-8"))["az_stage_runtime_sec"]
      for p in glob.glob("results/runs/*__n=2000__*.json")]
if rt:
    per_step = statistics.median(rt) / 2000
    print(f"measured: {statistics.median(rt):.0f} s/run -> {per_step:.3f} s/optimizer-step")
    print(f"full grid (295k step-equivalents), serial: {295000*per_step/3600:.1f} h")
    print("Replace the projections in docs/COMPUTE_ESTIMATES.md with this.")

In [ ]:
# Archive whatever this session produced.
import shutil
shutil.make_archive("/content/results_snapshot", "zip", "results")
from google.colab import files; files.download("/content/results_snapshot.zip")